# Semantic Chunking

Groups one PDF's unified document (dual-pipeline merged text + Stage 1
figure captions, from `image_understanding.ipynb`) into topically
coherent chunks, using embedding similarity rather than heading
structure or LLM-judged continuity.

Each `UnifiedItem` (a page's transcribed text, or a `[FIGURE:...]`
caption block) is embedded as one unit, in document order. A chunk
boundary is placed wherever the cosine distance between consecutive
item embeddings exceeds a percentile-based threshold over the whole
document — sections that drift topic sharply split apart; a figure
immediately following the text it illustrates stays in the same chunk.

See `src/ingestion/semantic_chunk.py::SemanticChunker`. Output chunks
are the unit fed to concept/relation extraction downstream.

## Setup

In [1]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent  # notebook lives in notebooks/
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from ingestion.semantic_chunk import SemanticChunker
from ingestion.unify import build_unified_items_from_merged, render_unified_markdown

PROJECT_ROOT

/Users/michaeleko/Documents/Works/aiml-institute/challenge-2/intelligent-tutoring-system-for-medical-student/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PosixPath('/Users/michaeleko/Documents/Works/aiml-institute/challenge-2/intelligent-tutoring-system-for-medical-student')

## Rebuild the unified document

`image_understanding.ipynb` writes the rendered text/markdown but not
the underlying `UnifiedItem` list, so it's rebuilt here from the two
artifacts that stage already produced: the dual-pipeline merged
document and the Stage 1 caption results.

In [2]:
PDF_STEM = "Anatomy of Neck - Basic of DEMN.pdf_origin"
OUTPUT_DIR = PROJECT_ROOT / "output" / PDF_STEM / "auto"
MERGED_PATH = OUTPUT_DIR / f"{PDF_STEM}_dual_pipeline_merged.json"
CAPTIONS_PATH = OUTPUT_DIR / f"{PDF_STEM}_stage1_captions.json"

with open(MERGED_PATH) as f:
    merged_document = json.load(f)

with open(CAPTIONS_PATH) as f:
    caption_results = json.load(f)

captions = {r["item_id"]: r["caption"] for r in caption_results}
unified_items = build_unified_items_from_merged(merged_document, captions)

len(unified_items), unified_items[0]

(152,
 UnifiedItem(item_id='page0#text', page_idx=0, content_type='text', text='# Anatomy of the Neck\n## Basic of DEMN System', text_level=None))

## Chunk

In [3]:
chunker = SemanticChunker()
chunks = chunker.chunk(unified_items, percentile=95.0, max_chars=6000)
chunker.unload()

len(chunks), [len(c) for c in chunks][:20]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11771.48it/s]


(38, [8, 2, 6, 3, 5, 5, 5, 1, 1, 6, 4, 4, 5, 5, 3, 1, 1, 4, 4, 5])

## Inspect a sample chunk

In [4]:
sample_chunk = chunks[len(chunks) // 2]
print(f"{len(sample_chunk)} items, item_ids: {[i.item_id for i in sample_chunk]}\n")
print(render_unified_markdown(sample_chunk))

5 items, item_ids: ['page32#text', 'Anatomy of Neck - Basic of DEMN.pdf_origin#p32#275', 'page33#text', 'page34#text', 'Anatomy of Neck - Basic of DEMN.pdf_origin#p34#290']

# Esophagus

## Cervical:
- Cervical begins at the lower end of pharynx (level of 6th vertebra or lower border of cricoid cartilage) and extends to the thoracic inlet (suprasternal notch); 18 cm from incisors.

## Thoracic:
- Upper thoracic: from thoracic inlet to level of tracheal bifurcation; 18-23 cm.
- Mid thoracic: from tracheal bifurcation midway to gastroesophageal junction; 24-32 cm.
- Lower thoracic: from midway between tracheal bifurcation and gastroesophageal junction to GE junction, including abdominal esophagus; 32-40 cm.

## Abdominal:
- Considered part of lower thoracic esophagus; 32-40 cm.

> [FIGURE:Anatomy of Neck - Basic of DEMN.pdf_origin#p32#275] The image is a diagram illustrating the anatomy and sections of the esophagus. It includes a side view of the esophagus with labeled parts and corresp

## Save chunks

Each chunk saved as its rendered markdown plus the item_ids it spans —
the item_ids are what a later concept-extraction stage would attach to
extracted concepts for content linkage back to source.

In [5]:
chunks_payload = [
    {
        "chunk_index": i,
        "item_ids": [item.item_id for item in chunk],
        "text": render_unified_markdown(chunk),
    }
    for i, chunk in enumerate(chunks)
]

chunks_path = OUTPUT_DIR / f"{PDF_STEM}_semantic_chunks.json"
with open(chunks_path, "w") as f:
    json.dump(chunks_payload, f, indent=2)

chunks_path

PosixPath('/Users/michaeleko/Documents/Works/aiml-institute/challenge-2/intelligent-tutoring-system-for-medical-student/output/Anatomy of Neck - Basic of DEMN.pdf_origin/auto/Anatomy of Neck - Basic of DEMN.pdf_origin_semantic_chunks.json')